# LIBRAS Dataset Exploration

This notebook explores the LIBRAS text-to-gloss dataset, analyzing:
- Dataset statistics
- Vocabulary distribution
- Sentence length analysis
- Gloss patterns
- Syntax tag distribution

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import numpy as np

from utils.data_loader import LibrasDataLoader
from utils.syntax_processor import PortugueseSyntaxProcessor

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Dataset

In [ ]:
# Load data
loader = LibrasDataLoader()
data = loader.load_libras_csv()

print(f"Total samples: {len(data)}")
print(f"\nSample data:")
for i in range(3):
    print(f"\n{i+1}. Source: {data[i]['source']}")
    print(f"   Target: {data[i]['target']}")

## 2. Basic Statistics

In [ ]:
# Convert to DataFrame
df = pd.DataFrame(data)

# Add length columns
df['source_len'] = df['source'].apply(lambda x: len(x.split()))
df['target_len'] = df['target'].apply(lambda x: len(x.split()))

print("Dataset Statistics:")
print("=" * 60)
print(f"Total samples: {len(df)}")
print(f"\nSource (PT-BR) length:")
print(df['source_len'].describe())
print(f"\nTarget (LIBRAS) length:")
print(df['target_len'].describe())

## 3. Length Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Source length distribution
axes[0].hist(df['source_len'], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Number of Words')
axes[0].set_ylabel('Frequency')
axes[0].set_title('PT-BR Sentence Length Distribution')
axes[0].axvline(df['source_len'].mean(), color='r', linestyle='--', label=f'Mean: {df["source_len"].mean():.1f}')
axes[0].legend()

# Target length distribution
axes[1].hist(df['target_len'], bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Number of Glosses')
axes[1].set_ylabel('Frequency')
axes[1].set_title('LIBRAS Gloss Length Distribution')
axes[1].axvline(df['target_len'].mean(), color='r', linestyle='--', label=f'Mean: {df["target_len"].mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Vocabulary Analysis

In [ ]:
# Source vocabulary
source_words = []
for text in df['source']:
    source_words.extend(text.lower().split())

source_vocab = Counter(source_words)

# Target vocabulary
target_glosses = []
for text in df['target']:
    target_glosses.extend(text.split())

target_vocab = Counter(target_glosses)

print(f"Source (PT-BR) vocabulary size: {len(source_vocab)}")
print(f"Target (LIBRAS) vocabulary size: {len(target_vocab)}")
print(f"\nMost common PT-BR words:")
for word, count in source_vocab.most_common(10):
    print(f"  {word}: {count}")

print(f"\nMost common LIBRAS glosses:")
for gloss, count in target_vocab.most_common(10):
    print(f"  {gloss}: {count}")

## 5. Length Correlation

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df['source_len'], df['target_len'], alpha=0.3)
plt.xlabel('PT-BR Sentence Length (words)')
plt.ylabel('LIBRAS Gloss Length (glosses)')
plt.title('Correlation between Source and Target Lengths')

# Add trend line
z = np.polyfit(df['source_len'], df['target_len'], 1)
p = np.poly1d(z)
plt.plot(df['source_len'], p(df['source_len']), "r--", alpha=0.8, label=f'Trend: y={z[0]:.2f}x+{z[1]:.2f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

correlation = df['source_len'].corr(df['target_len'])
print(f"Correlation coefficient: {correlation:.3f}")

## 6. Syntax Analysis

In [ ]:
# Process syntax for sample
processor = PortugueseSyntaxProcessor()
sample_size = min(1000, len(data))
sample_data = data[:sample_size]

print(f"Processing syntax for {sample_size} samples...")
sources = [item['source'] for item in sample_data]
tokens_list, dep_tags_list = processor.process_batch(sources)

# Get tag statistics
tag_stats = processor.get_tag_statistics(dep_tags_list)

# Plot top tags
top_tags = sorted(tag_stats.items(), key=lambda x: x[1], reverse=True)[:15]
tags, counts = zip(*top_tags)

plt.figure(figsize=(12, 6))
plt.barh(tags, counts)
plt.xlabel('Count')
plt.ylabel('Dependency Tag')
plt.title(f'Top 15 Dependency Tags (sample of {sample_size} sentences)')
plt.tight_layout()
plt.show()

## 7. Save Statistics

In [ ]:
statistics = {
    'total_samples': len(df),
    'source_vocab_size': len(source_vocab),
    'target_vocab_size': len(target_vocab),
    'avg_source_length': float(df['source_len'].mean()),
    'avg_target_length': float(df['target_len'].mean()),
    'length_correlation': float(correlation),
    'top_source_words': source_vocab.most_common(20),
    'top_target_glosses': target_vocab.most_common(20),
}

with open('../data/processed/statistics.json', 'w', encoding='utf-8') as f:
    json.dump(statistics, f, indent=2, ensure_ascii=False)

print("Statistics saved to data/processed/statistics.json")